# CLF2 — Intent Classifier Test
Tests: `scheduling_add`, `scheduling_delete`, `scheduling_update`, `scheduling_show`, `learning_create`, `learning_delete`, `learning_update`, `behavioral`

In [ ]:
import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

MODEL_PATH = r'D:\FYP_20March\Eva_AI_Server\models\clf2_model'

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_PATH)
model     = DistilBertForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()

id2label = model.config.id2label
print('Model loaded.')
print('Labels:', id2label)

In [ ]:
# ── Add your test inputs here ──────────────────────────────────────
inputs = [
    "add physics exam on monday at 3pm",
    "delete math homework",
    "reschedule phy exam to friday",
    "show my tasks for tomorrow",
    "create a python roadmap for 2 weeks",
    "delete my javascript learning plan",
    "pause my python learning",
    "how many hours did I study today",
    "learn django starting next monday",
    "mark physics exam as done",
]

In [ ]:
print(f"{'Input':<50} {'Predicted':<25} {'Confidence'}")
print('-' * 90)

for text in inputs:
    enc = tokenizer(
        text.lower(),
        return_tensors = 'pt',
        truncation     = True,
        max_length     = 128,
    )
    with torch.no_grad():
        logits = model(**enc).logits
    probs      = torch.softmax(logits, dim=-1).squeeze()
    pred_id    = probs.argmax().item()
    pred_label = id2label[pred_id]
    confidence = probs[pred_id].item()
    print(f"{text:<50} {pred_label:<25} {confidence:.4f}")

In [ ]:
# ── Detailed probabilities for all classes ─────────────────────────
text = inputs[0]   # change index to inspect a specific input

enc = tokenizer(text.lower(), return_tensors='pt', truncation=True, max_length=128)
with torch.no_grad():
    logits = model(**enc).logits
probs = torch.softmax(logits, dim=-1).squeeze()

print(f'Input: "{text}"\n')
for i, prob in enumerate(sorted(zip(probs.tolist(), [id2label[j] for j in range(len(id2label))]), reverse=True)):
    print(f'  {prob[1]:<25}: {prob[0]:.4f}')